## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **1113bdc3-8fd5-4c5e-a5b2-3e074f1b1e68**


## Dataset

# **Agglo - Réserves naturelles protégées**


## Description

Réserves naturelles et biologiques en vertu du droit cantonal ou national et gérés par des collectivités ou par des associations de protection Périmètre Grand Genève et régions voisines &#40;extraction&#41; Les réserves naturelles servent de refuge à la grande majorité des espèces animales et végétales et abritent des biotopes uniques et irremplacables. L&#x27;utilisation des lieux, les constructions, la circulation et les activités de loisirs y sont soumises à des restrictions afin de garantir le maintien de ces valeurs naturelles. Le Grand Genève accueille des réserves de différents types : - Canton de Vaud : districts francs fédéraux selon la loi fédérale de 1988 sur la chasse et la protection des mammifères et oiseaux sauvages et l&#x27;ordonnance sur les districts francs; - France : réserves biologiques forestières : espaces, en forêts relevant du régime forestier, classés volontairement par l&#x27;ONF comme réserves biologiques intégrales &#40;RBI&#41; et réserves biologiques dirigées &#40;RBD&#41;; - Canton de Vaud : réserves de faune; - Canton de Vaud : réserves forestières constituant un réseau mis en place par le canton, avec le soutien financier de la Confédération; - Canton de Genève : réserves naturelles définies sur la base du règlement sur la protection du paysage, des milieux naturels et de la flore; - Canton de Vaud : réserves naturelles de statut public &#40;les réserves naturelles de droit privé avec droits réels ou sans convention figurent dans deux autres inventaires diffusables avec restriction&#41;; - France : réserves naturelles nationales créées par l&#x27;État, espace fortement réglementé, soustrait à toute intervention artificielle susceptible de le dégrader et qui bénéficie d&#x27;un plan de gestion et pour certaines d&#x27;une superficie additionnelle en « zone tampon » protégée par ce dispositif - France : réserves naturelles régionales créées par les régions et présentent les mêmes caractéristiques de gestion que les réserves naturelles nationales.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/agglo-reserves-naturelles-protegees)


## Metadata

- **identifier:** AGGLO&#95;NAT&#95;PERI&#95;PRO&#95;RESERVES@canton-geneve
- **publisher.name:** Direction du projet d&#x27;agglomération Grand Genève
- **organization.display&#95;name:** Kanton Genf
- **organization.url:** 
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** 
- **issued:** 2025-06-12 00:00:00
- **metadata&#95;created:** 2026-01-12T16:01:04.438174
- **metadata&#95;modified:** 2026-11-11T03:01:41.779038
- **Resource id:** 280e3855-be0d-4a33-a5fe-0419d73ac74b
- **Resource name:** CSV
- **Resource package&#95;id:** 1113bdc3-8fd5-4c5e-a5b2-3e074f1b1e68
- **Resource description:** Service téléchargement extracteur
- **Resource issued:** 2025-06-12 00:00:00
- **Resource modified:** 2026-10-29 00:00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;open
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;open


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **CSV** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://extract.sitg.ge.ch/files/AGGLO_NAT_PERI_PRO_RESERVES-csv.zip")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** louca.lerch@etat.ge.ch &#124; Direction du projet d&#x27;agglomération Grand Genève
